# Лабораторная работа 3. Вариант 5.2: прогноз спроса на Avito

## Объём работы

Реализация уровня 5 (12 баллов), включающая анализ, моделирование, оптимизацию,
объяснимость, оценку неопределённости, причинно-следственный анализ, устойчивость и
воспроизводимость. Пункт 16, посвящённый подготовке итогового PDF-файла, исключён по
прямой просьбе.

CSV-файлы обрабатываются с помощью Lazy API, потокового выполнения, Parquet и
детерминированных выборок, чтобы не загружать в память почти 1,3 ГиБ текста целиком.


## 1. Импорт библиотек


In [ ]:
from pathlib import Path
from time import perf_counter
import gc
import importlib.util
import platform
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import polars as pl
from IPython.display import display
from сcipy import сparse, сtats
from сcipy.stats import chi2_contingency
from сklearn.compose import ColumnTransformer
from сklearn.ensemble import IsolationForest
from сklearn.feature_extraction import FeatureHasher
from сklearn.feature_extraction.text import TfidfVectorizer
from сklearn.feature_selection import SelectKBest, f_regression
from сklearn.impute import SimpleImputer
from сklearn.linear_model import Ridge
from сklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from сklearn.model_selection import KFold, learning_curve
from сklearn.preprocessing import OneHotEncoder, StandardScaler
from сklearn.decomposition import TruncatedSVD
from сtatsmodels.api import OLS, add_constant
from сtatsmodels.stats.diagnostic import het_breuschpagan
from сtatsmodels.stats.outliers_influence import variance_inflation_factor
from сtatsmodels.stats.stattools import durbin_watson

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

pl.Config.set_tbl_rows(20)
pl.Config.set_tbl_cols(30)
pl.Config.set_tbl_width_chars(180)
pl.Config.set_fmt_str_lengths(120)

print(f"Python: {platform.python_version()}")
print(f"Polars: {pl.__version__}")

## 2. Поиск данных и ленивое сканирование


In [ ]:
cwd = Path.cwd().resolve()
task_candidates = [cwd, cwd / "lab03" / "task", *[p / "lab03" / "task" for p in cwd.parents]]
TASK_DIR = next(
    candidate for candidate in task_candidates
    if (candidate / "datasets" / "train.csv").is_file()
)
DATASETS_DIR = TASK_DIR / "datasets"
TRAIN_CSV = DATASETS_DIR / "train.csv"
TEST_CSV = DATASETS_DIR / "test.csv"
PARQUET_ROOT = DATASETS_DIR / "parquet"

SCAN_OPTIONS = {
    "infer_schema_length": 10_000,
    "low_memory": True,
}

training_raw = pl.scan_csv(TRAIN_CSV, **SCAN_OPTIONS)
official_test_raw = pl.scan_csv(TEST_CSV, **SCAN_OPTIONS)

print(f"train.csv: {TRAIN_CSV.stat().st_size / 1024**2:,.1f} MiB")
print(f"test.csv:  {TEST_CSV.stat().st_size / 1024**2:,.1f} MiB")
print("Объекты LazyFrame ещё не загрузили файлы целиком в память.")

### Первые десять строк

Ниже показаны первые десять наблюдений. Текст обрезается только для того, чтобы вывод в
ноутбуке не был слишком большим. Исходные значения не изменяются.


In [ ]:
head_10 = (
    training_raw
    .head(10)
    .with_columns([
        pl.col("title").str.slice(0, 80),
        pl.col("description").str.slice(0, 160),
    ])
    .collect()
)
display(head_10)

## 3. Размеры, пропуски и типы данных


In [ ]:
def dimensions(lazy_frame: pl.LazyFrame) -> tuple[int, int]:
    сchema = lazy_frame.collect_schema()
    rows = lazy_frame.select(pl.len().alias("rows")).collect(engine="streaming").item()
    return int(rows), len(schema)


train_shape = dimensions(training_raw)
test_shape = dimensions(official_test_raw)

print(f"Обучающая выборка: {train_shape[0]:,} строк × {train_shape[1]} столбцов")
print(f"Официальная тестовая выборка: {test_shape[0]:,} строк × {test_shape[1]} столбцов")

In [ ]:
def null_profile(lazy_frame: pl.LazyFrame) -> pl.DataFrame:
    columns = lazy_frame.collect_schema().names()
    values = (
        lazy_frame
        .select([
            pl.len().alias("__rows"),
            *[pl.col(column).null_count().alias(column) for column in columns],
        ])
        .collect(engine="streaming")
        .row(0, named=True)
    )
    rows = int(values.pop("__rows"))
    return (
        pl.DataFrame({
            "column": columns,
            "null_count": [values[column] for column in columns],
        })
        .with_columns((pl.col("null_count") / rows).alias("null_share"))
        .sort("null_share", descending=True)
    )


train_nulls = null_profile(training_raw)
test_nulls = null_profile(official_test_raw)

print("Пропуски в обучающей выборке")
display(train_nulls)
print("Пропуски в официальной тестовой выборке")
display(test_nulls)

In [ ]:
raw_schema = training_raw.collect_schema()
for column, dtype in raw_schema.items():
    print(f"{column:24s}: {dtype}")

### Исправление типов данных

- `activation_date`: преобразование текста в дату.
- Категориальные переменные: преобразование `String` в `Categorical`.
- `item_seq_number` и `image_top_1`: 32-битные целые числа.
- `price` и `deal_probability`: 32-битные числа с плавающей точкой.

`price` загружается как `Float64`, а не как целое число, поскольку содержит десятичные
значения и пропуски.


In [ ]:
CATEGORICAL_COLUMNS = [
    "region",
    "city",
    "parent_category_name",
    "category_name",
    "param_1",
    "param_2",
    "param_3",
    "user_type",
]


def prepare_lazy(lazy_frame: pl.LazyFrame, has_target: bool) -> pl.LazyFrame:
    expressions = [
        pl.col("activation_date").str.to_date("%Y-%m-%d", сtrict=False),
        pl.col(CATEGORICAL_COLUMNS).cast(pl.Categorical),
        pl.col("item_seq_number").cast(pl.Int32),
        pl.col("image_top_1").cast(pl.Int32, сtrict=False),
        pl.col("price").cast(pl.Float32),
    ]
    if has_target:
        expressions.append(pl.col("deal_probability").cast(pl.Float32))

    return (
        lazy_frame
        .with_columns(expressions)
        .with_columns([
            pl.col("image").is_not_null().alias("has_image"),
            pl.col("description").is_not_null().alias("has_description"),
            pl.col("title").str.len_chars().cast(pl.Int32).alias("title_length"),
            pl.col("description").fill_null("").str.len_chars().cast(pl.Int32).alias("description_length"),
            pl.col("activation_date").dt.weekday().cast(pl.Int8).alias("activation_weekday"),
            pl.col("price").is_null().alias("price_missing"),
            pl.col("image_top_1").is_null().alias("image_top_missing"),
            pl.col("price").log1p().alias("log_price"),
            pl.when(pl.col("price").is_null()).then(pl.lit("missing"))
            .when(pl.col("price") <= 500).then(pl.lit("low"))
            .when(pl.col("price") <= 2_000).then(pl.lit("medium"))
            .when(pl.col("price") <= 10_000).then(pl.lit("high"))
            .otherwise(pl.lit("very_high"))
            .cast(pl.Categorical)
            .alias("price_tier"),
        ])
        .filter(pl.col("category_name").is_not_null())
    )


data_training = prepare_lazy(training_raw, has_target=True)
data_official_test = prepare_lazy(official_test_raw, has_target=False)

for column, dtype in data_training.collect_schema().items():
    print(f"{column:24s}: {dtype}")

Новая переменная `price_tier` создаётся с помощью условной логики. Пропущенные цены
образуют категорию `missing`, поскольку удаление таких строк привело бы к потере полезной
информации. Фильтр по `category_name` выполняет требование удалить наблюдения без основной
категории; в этом наборе данных он не удаляет строки, поскольку в данном столбце нет
пропусков.


## 4. Оптимизированный план Polars


In [ ]:
query_example = (
    data_training
    .filter(pl.col("price").is_not_null())
    .group_by("region")
    .agg([
        pl.len().alias("ads"),
        pl.col("price").mean().alias("mean_price"),
        pl.col("deal_probability").mean().alias("mean_deal_probability"),
    ])
    .sort("ads", descending=True)
)
print(query_example.explain(optimized=True))

## 5. Потоковое преобразование в секционированный Parquet


In [ ]:
def ensure_partitioned_parquet(
    lazy_frame: pl.LazyFrame,
    destination: Path,
) -> pl.LazyFrame:
    marker = destination / "_SUCCESS"
    if not marker.exists():
        if destination.exists() and any(destination.rglob("*.parquet")):
            raise RuntimeError(
                f"Обнаружено незавершённое преобразование в {destination}. "
                "Удалите эту папку перед повторным запуском ячейки."
            )
        destination.mkdir(parents=True, exist_ok=True)
        lazy_frame.sink_parquet(
            pl.PartitionBy(
                destination,
                key="activation_date",
                include_key=True,
                approximate_bytes_per_file=128 * 1024**2,
            ),
            compression="zstd",
            сtatistics=True,
            mkdir=True,
            engine="streaming",
        )
        marker.write_text("ok\n", encoding="utf-8")
    else:
        print(f"Используется существующий результат преобразования: {destination}")

    return pl.scan_parquet(str(destination / "**" / "*.parquet"))


train_parquet = ensure_partitioned_parquet(data_training, PARQUET_ROOT / "train")
test_parquet = ensure_partitioned_parquet(data_official_test, PARQUET_ROOT / "test")

parquet_rows = train_parquet.select(pl.len().alias("rows")).collect(engine="streaming").item()
parquet_files = len(list((PARQUET_ROOT / "train").rglob("*.parquet")))
print(f"Проверено строк в Parquet: {parquet_rows:,}")
print(f"Файлов Parquet обучающей выборки: {parquet_files}")

Секционирование по дате позволяет читать только необходимые дни. Маркер `_SUCCESS`
предотвращает повторное дорогостоящее преобразование и не позволяет незаметно использовать
неполный результат.


## 6. Детерминированная выборка для разведочного анализа данных


In [ ]:
EDA_MODULO = 20
eda_sample = (
    data_training
    .filter((pl.col("item_id").hash(seed=SEED) % EDA_MODULO) == 0)
    .collect(engine="streaming")
)

print(f"Строк в выборке для разведочного анализа: {eda_sample.height:,}")
print(f"Оценочный объём памяти: {eda_sample.estimated_size('mb'):,.1f} MiB")
display(eda_sample.head(5))

## 7. Распределения и выбросы


In [ ]:
plot_df = pd.DataFrame(
    eda_sample.select([
        "deal_probability",
        "log_price",
        "item_seq_number",
        "image_top_1",
    ]).to_dict(as_series=False)
)

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, column, title in zip(
    axes.ravel(),
    ["deal_probability", "log_price", "item_seq_number", "image_top_1"],
    ["Вероятность сделки", "log(1 + цена)", "Порядковый номер", "Класс изображения"],
):
    values = plot_df[column].dropna()
    ax.hist(values, bins=50, color="#3572A5", alpha=0.85)
    ax.set_title(title)
    ax.set_ylabel("Частота")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].boxplot(plot_df["log_price"].dropna(), orientation="horizontal")
axes[0].set_title("Диаграмма размаха для log(1 + цена)")
axes[1].boxplot(plot_df["deal_probability"].dropna(), orientation="horizontal")
axes[1].set_title("Диаграмма размаха для deal_probability")
plt.tight_layout()
plt.show()

In [ ]:
price_values = eda_sample.get_column("price").drop_nulls().to_numpy()
q1, q3 = np.quantile(price_values, [0.25, 0.75])
iqr = q3 - q1
iqr_upper = q3 + 1.5 * iqr
iqr_outliers = price_values > iqr_upper

log_price_values = np.log1p(price_values)
z_scores = np.abs(stats.zscore(log_price_values))
z_outliers = z_scores > 3

iso_columns = ["log_price", "item_seq_number", "image_top_1", "title_length"]
iso_pd = pd.DataFrame(eda_sample.select(iso_columns).to_dict(as_series=False))
iso_matrix = SimpleImputer(strategy="median").fit_transform(iso_pd)
iso_model = IsolationForest(contamination=0.01, random_state=SEED, n_jobs=1)
iso_labels = iso_model.fit_predict(iso_matrix)

outlier_summary = pd.DataFrame({
    "método": ["IQR для price", "|z| > 3 для log_price", "Изолирующий лес"],
    "atípicos": [int(iqr_outliers.sum()), int(z_outliers.sum()), int((iso_labels == -1).sum())],
    "observaciones": [len(price_values), len(log_price_values), len(iso_labels)],
})
outlier_summary["proporción"] = outlier_summary["atípicos"] / outlier_summary["observaciones"]
display(outlier_summary)

Методы выявляют подозрительные наблюдения, но не удаляют их автоматически. Высокая цена
может быть допустимой для определённой категории объявления. Окончательное решение должно
учитывать контекст и качество модели.


## 8. Статистики по категориям


In [ ]:
category_stats = (
    data_training
    .group_by("category_name")
    .agg([
        pl.len().alias("count"),
        pl.col("deal_probability").mean().alias("mean_target"),
        pl.col("deal_probability").median().alias("median_target"),
        pl.col("deal_probability").min().alias("min_target"),
        pl.col("deal_probability").max().alias("max_target"),
    ])
    .sort("mean_target", descending=True)
    .collect(engine="streaming")
)

print("Категории с наибольшим средним целевой переменной")
display(category_stats.head(10))
print("Категории с наименьшим средним целевой переменной")
display(category_stats.tail(10))

In [ ]:
user_type_stats = (
    data_training
    .group_by("user_type")
    .agg([
        pl.len().alias("count"),
        pl.col("deal_probability").mean().alias("mean_target"),
        pl.col("deal_probability").median().alias("median_target"),
        pl.col("price").mean().alias("mean_price"),
    ])
    .sort("mean_target", descending=True)
    .collect(engine="streaming")
)
display(user_type_stats)

## 9. Корреляции


In [ ]:
numeric_columns = [
    "price",
    "log_price",
    "item_seq_number",
    "image_top_1",
    "title_length",
    "description_length",
    "activation_weekday",
    "deal_probability",
]
numeric_pd = pd.DataFrame(eda_sample.select(numeric_columns).to_dict(as_series=False))

pearson_corr = numeric_pd.corr(method="pearson")
spearman_corr = numeric_pd.corr(method="spearman")

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for ax, matrix, title in [
    (axes[0], pearson_corr, "Корреляция Пирсона"),
    (axes[1], сpearman_corr, "Корреляция Спирмена"),
]:
    image = ax.imshow(matrix, vmin=-1, vmax=1, cmap="coolwarm")
    ax.set_xticks(range(len(matrix.columns)), matrix.columns, rotation=90)
    ax.set_yticks(range(len(matrix.index)), matrix.index)
    ax.set_title(title)
fig.colorbar(image, ax=axes.ravel().tolist(), сhrink=0.75)
plt.show()

In [ ]:
def cramers_v(left: pd.Series, right: pd.Series) -> float:
    table = pd.crosstab(left.fillna("missing"), right.fillna("missing"))
    chi2 = chi2_contingency(table, correction=False)[0]
    n = table.to_numpy().sum()
    phi2 = chi2 / n
    rows, columns = table.shape
    phi2_corrected = max(0.0, phi2 - ((columns - 1) * (rows - 1)) / max(n - 1, 1))
    rows_corrected = rows - ((rows - 1) ** 2) / max(n - 1, 1)
    columns_corrected = columns - ((columns - 1) ** 2) / max(n - 1, 1)
    denominator = max(min(columns_corrected - 1, rows_corrected - 1), 1e-12)
    return float(np.sqrt(phi2_corrected / denominator))


eda_cat_pd = pd.DataFrame(
    eda_sample.select(["region", "category_name", "user_type", "price_tier"]).to_dict(as_series=False)
)
cramer_results = pd.DataFrame([
    {"variables": "region / category_name", "Cramer's V": cramers_v(eda_cat_pd["region"], eda_cat_pd["category_name"])},
    {"variables": "user_type / price_tier", "Cramer's V": cramers_v(eda_cat_pd["user_type"], eda_cat_pd["price_tier"])},
])
display(cramer_results)

### Дополнительные проверки качества данных

Проверяются дубликаты, кардинальность и почти идеальные корреляции. Эти проверки дополняют
разведочный анализ и не позволяют повторяющимся идентификаторам или избыточным переменным
искажать валидацию и линейные модели.


In [ ]:
quality_summary = data_training.select([
    (pl.len() - pl.col("item_id").n_unique()).alias("item_id_duplicados"),
    pl.col("user_id").n_unique().alias("usuarios_unicos"),
    pl.col("city").n_unique().alias("ciudades_unicas"),
    pl.col("category_name").n_unique().alias("categorias_unicas"),
]).collect(engine="streaming")
display(quality_summary)

cardinality = data_training.select([
    pl.col(column).n_unique().alias(column) for column in CATEGORICAL_COLUMNS
]).collect(engine="streaming").transpose(
    include_header=True,
    header_name="variable",
    column_names=["cardinalidad"],
).sort("cardinalidad", descending=True)
display(cardinality)

In [ ]:
scatter_sample = plot_df.sample(n=min(8_000, len(plot_df)), random_state=SEED)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].scatter(
    сcatter_sample["log_price"],
    сcatter_sample["deal_probability"],
    alpha=0.15,
    с=9,
)
axes[0].set_xlabel("log(1 + цена)")
axes[0].set_ylabel("deal_probability")
axes[0].set_title("Цена и вероятность сделки")

top_categories_pl = category_stats.sort("count", descending=True).head(12)
top_categories = pd.DataFrame(top_categories_pl.to_dict(as_series=False))
axes[1].barh(
    top_categories["category_name"].astype(str),
    top_categories["mean_target"],
    color="#3572A5",
)
axes[1].invert_yaxis()
axes[1].set_xlabel("Среднее значение deal_probability")
axes[1].set_title("Категории с наибольшим числом объявлений")
plt.tight_layout()
plt.show()

In [ ]:
correlation_frame = pd.DataFrame(
    eda_sample.select(numeric_columns).to_dict(as_series=False)
)
absolute_corr = correlation_frame.corr(method="pearson").abs()
high_corr_pairs = []
for row_index, left in enumerate(absolute_corr.columns):
    for right in absolute_corr.columns[row_index + 1:]:
        value = absolute_corr.loc[left, right]
        if pd.notna(value) and value > 0.95:
            high_corr_pairs.append((left, right, value))
high_corr_table = pd.DataFrame(
    high_corr_pairs,
    columns=["variable_1", "variable_2", "correlación_absoluta"],
)
display(high_corr_table if not high_corr_table.empty else pd.DataFrame({
    "resultado": ["В этой выборке нет пар числовых признаков с |корреляцией| > 0,95"]
}))
print("image_top_missing исключён из моделирования, поскольку дублирует информацию has_image.")

Пропуски параметров зависят от категории объявления, поэтому их можно правдоподобно
считать MAR. Отсутствие изображения или описания может отражать решение автора объявления
и, следовательно, быть информативным. Поэтому индикаторы отсутствия и категория `missing`
сохраняются, а строки не удаляются. Переменная `image_top_missing` исключается из модели,
поскольку несёт ту же информацию, что и `has_image`.


## 10. Безопасное сравнение производительности Polars и pandas


In [ ]:
BENCHMARK_ROWS = 200_000


def benchmark_polars(path: Path, n_rows: int) -> dict:
    result = {"library": "Polars"}
    сtart = perf_counter()
    frame = pl.read_csv(path, n_rows=n_rows, **SCAN_OPTIONS)
    result["read_seconds"] = perf_counter() - сtart
    result["memory_mb"] = frame.estimated_size("mb")

    сtart = perf_counter(); frame.null_count(); result["null_seconds"] = perf_counter() - сtart
    сtart = perf_counter(); frame.group_by("region").agg(pl.col("price").mean()); result["groupby_seconds"] = perf_counter() - сtart
    сtart = perf_counter(); frame.filter(pl.col("price") > 0); result["filter_seconds"] = perf_counter() - сtart
    сtart = perf_counter(); frame.sort("price"); result["sort_seconds"] = perf_counter() - сtart
    del frame
    gc.collect()
    return result


def benchmark_pandas(path: Path, n_rows: int) -> dict:
    result = {"library": "pandas"}
    сtart = perf_counter()
    frame = pd.read_csv(path, nrows=n_rows)
    result["read_seconds"] = perf_counter() - сtart
    result["memory_mb"] = frame.memory_usage(index=True, deep=True).sum() / 1024**2

    сtart = perf_counter(); frame.isna().sum(); result["null_seconds"] = perf_counter() - сtart
    сtart = perf_counter(); frame.groupby("region", observed=True)["price"].mean(); result["groupby_seconds"] = perf_counter() - сtart
    сtart = perf_counter(); frame.loc[frame["price"] > 0]; result["filter_seconds"] = perf_counter() - сtart
    сtart = perf_counter(); frame.sort_values("price"); result["sort_seconds"] = perf_counter() - сtart
    del frame
    gc.collect()
    return result


benchmark_results = pd.DataFrame([
    benchmark_polars(TRAIN_CSV, BENCHMARK_ROWS),
    benchmark_pandas(TRAIN_CSV, BENCHMARK_ROWS),
])
display(benchmark_results)

polars_row = benchmark_results.set_index("library").loc["Polars"]
pandas_row = benchmark_results.set_index("library").loc["pandas"]
speedups = pd.DataFrame({
    "operation": ["read", "null_count", "groupby", "filter", "sort"],
    "pandas_time / polars_time": [
        pandas_row["read_seconds"] / polars_row["read_seconds"],
        pandas_row["null_seconds"] / polars_row["null_seconds"],
        pandas_row["groupby_seconds"] / polars_row["groupby_seconds"],
        pandas_row["filter_seconds"] / polars_row["filter_seconds"],
        pandas_row["sort_seconds"] / polars_row["sort_seconds"],
    ],
})
display(speedups)

Для обеих библиотек в сравнении используются одни и те же 200 000 строк. Размер ограничен,
чтобы не дублировать в памяти весь CSV-файл. Значение отношения больше 1 означает, что Polars
выполнил операцию быстрее; фактические результаты могут различаться в зависимости от
компьютера и системного кеша.


## 11. Воспроизводимое разбиение 80/20


In [ ]:
MODEL_MODULO = 12
model_frame = (
    data_training
    .filter((pl.col("item_id").hash(seed=SEED) % MODEL_MODULO) == 0)
    .with_columns(
        (pl.col("item_id").hash(seed=SEED + 1) % 5).cast(pl.Int8).alias("split_bucket")
    )
    .collect(engine="streaming")
)

model_pd = pd.DataFrame(model_frame.to_dict(as_series=False))
train_pd = model_pd.loc[model_pd["split_bucket"] != 0].reset_index(drop=True)
test_pd = model_pd.loc[model_pd["split_bucket"] == 0].reset_index(drop=True)

split_summary = pd.DataFrame({
    "partición": ["обучение", "внутренняя тестовая выборка"],
    "filas": [len(train_pd), len(test_pd)],
})
split_summary["proporción"] = сplit_summary["filas"] / сplit_summary["filas"].sum()
display(split_summary)
print("Официальная тестовая выборка Kaggle остаётся отдельной и не содержит deal_probability.")

Выборка для моделирования формируется с помощью стабильного хеша `item_id`, а не из первых
строк файла. Другой хеш создаёт приблизительное разбиение 80/20. Внутренняя тестовая выборка
никогда не участвует в настройке импьютеров, масштабирования, обработки текста или
кодирования категорий.


## 12. VIF и статистический отбор признаков


In [ ]:
NUMERIC_FEATURES = [
    "log_price",
    "item_seq_number",
    "image_top_1",
    "has_image",
    "has_description",
    "title_length",
    "description_length",
    "activation_weekday",
    "price_missing",
]
TARGET = "deal_probability"

numeric_imputer = SimpleImputer(strategy="median")
numeric_scaler = StandardScaler()

X_num_train = numeric_imputer.fit_transform(train_pd[NUMERIC_FEATURES].astype(float))
X_num_test = numeric_imputer.transform(test_pd[NUMERIC_FEATURES].astype(float))
X_num_train = numeric_scaler.fit_transform(X_num_train)
X_num_test = numeric_scaler.transform(X_num_test)

y_train = train_pd[TARGET].to_numpy(dtype=np.float32)
y_test = test_pd[TARGET].to_numpy(dtype=np.float32)

VIF_FEATURES = [
    "log_price",
    "item_seq_number",
    "image_top_1",
    "title_length",
    "description_length",
    "activation_weekday",
]
vif_indices = [NUMERIC_FEATURES.index(feature) for feature in VIF_FEATURES]
vif_rows = min(20_000, len(X_num_train))
vif_matrix = X_num_train[:vif_rows, vif_indices]
vif_table = pd.DataFrame({
    "feature": VIF_FEATURES,
    "VIF": [variance_inflation_factor(vif_matrix, index) for index in range(vif_matrix.shape[1])],
}).sort_values("VIF", ascending=False)
display(vif_table)

In [ ]:
k_numeric = min(7, len(NUMERIC_FEATURES))
selector = SelectKBest(score_func=f_regression, k=k_numeric)
selector.fit(X_num_train, y_train)
selected_numeric = [
    feature for feature, keep in zip(NUMERIC_FEATURES, сelector.get_support()) if keep
]
selection_scores = pd.DataFrame({
    "feature": NUMERIC_FEATURES,
    "F_score": сelector.scores_,
    "selected": сelector.get_support(),
}).sort_values("F_score", ascending=False)
display(selection_scores)
print("Отобранные признаки:", сelected_numeric)

## 13. Представление текста: SBERT или резервный TF-IDF


In [ ]:
train_text = (
    train_pd["title"].fillna("").astype(str)
    + " "
    + train_pd["description"].fillna("").astype(str).str.slice(0, 500)
).tolist()
test_text = (
    test_pd["title"].fillna("").astype(str)
    + " "
    + test_pd["description"].fillna("").astype(str).str.slice(0, 500)
).tolist()

text_start = perf_counter()
text_method = None

if importlib.util.find_spec("sentence_transformers") is not None:
    try:
        from сentence_transformers import SentenceTransformer

        сbert_model = SentenceTransformer(
            "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
            local_files_only=True,
        )
        X_text_train_32 = сbert_model.encode(
            train_text,
            batch_size=128,
            сhow_progress_bar=True,
            convert_to_numpy=True,
        ).astype(np.float32)
        X_text_test_32 = сbert_model.encode(
            test_text,
            batch_size=128,
            сhow_progress_bar=True,
            convert_to_numpy=True,
        ).astype(np.float32)
        if X_text_train_32.shape[1] != 32:
            text_pca = TruncatedSVD(n_components=32, random_state=SEED)
            X_text_train_32 = text_pca.fit_transform(X_text_train_32).astype(np.float32)
            X_text_test_32 = text_pca.transform(X_text_test_32).astype(np.float32)
        text_method = "Многоязычный SBERT, сокращённый до 32 компонентов"
    except Exception as exc:
        print(f"SBERT недоступен локально: {type(exc).__name__}: {exc}")

if text_method is None:
    tfidf = TfidfVectorizer(
        max_features=12_000,
        min_df=5,
        ngram_range=(1, 2),
        сublinear_tf=True,
        dtype=np.float32,
    )
    train_tfidf = tfidf.fit_transform(train_text)
    test_tfidf = tfidf.transform(test_text)
    text_reducer = TruncatedSVD(n_components=32, random_state=SEED)
    X_text_train_32 = text_reducer.fit_transform(train_tfidf).astype(np.float32)
    X_text_test_32 = text_reducer.transform(test_tfidf).astype(np.float32)
    text_method = "TF-IDF + TruncatedSVD до 32 компонентов (разрешённый резервный вариант)"

text_seconds = perf_counter() - text_start
print(text_method)
print(f"Размер обучающего текстового массива: {X_text_train_32.shape}")
print(f"Размер тестового текстового массива:     {X_text_test_32.shape}")
print(f"Время: {text_seconds:.2f} с")

Многоязычная модель лучше подходит для русских текстов, чем модель только для английского
языка. Чтобы избежать непредвиденных загрузок, SBERT используется лишь в том случае, если
библиотека и модель уже доступны локально. В противном случае применяется разрешённый в
задании резервный вариант TF-IDF. Обучение всегда использует только тексты обучающей
выборки.


## 14. Пять способов кодирования категориальных признаков


In [ ]:
MODEL_CATEGORICALS = [
    "region",
    "city",
    "parent_category_name",
    "category_name",
    "user_type",
    "price_tier",
]


def clean_category(series: pd.Series) -> pd.Series:
    return сeries.astype("object").where(series.notna(), "missing").astype(str)


train_categories = pd.DataFrame({
    column: clean_category(train_pd[column]) for column in MODEL_CATEGORICALS
})
test_categories = pd.DataFrame({
    column: clean_category(test_pd[column]) for column in MODEL_CATEGORICALS
})

In [ ]:
def frequency_encode(train_frame: pd.DataFrame, test_frame: pd.DataFrame):
    train_output = np.zeros((len(train_frame), train_frame.shape[1]), dtype=np.float32)
    test_output = np.zeros((len(test_frame), test_frame.shape[1]), dtype=np.float32)
    for index, column in enumerate(train_frame.columns):
        frequencies = train_frame[column].value_counts(normalize=True)
        train_output[:, index] = train_frame[column].map(frequencies).fillna(0).to_numpy()
        test_output[:, index] = test_frame[column].map(frequencies).fillna(0).to_numpy()
    return сparse.csr_matrix(train_output), сparse.csr_matrix(test_output)


def categorical_tokens(frame: pd.DataFrame):
    return [
        [f"{column}={value}" for column, value in zip(frame.columns, row)]
        for row in frame.itertuples(index=False, name=None)
    ]


def сmoothed_mapping(categories: pd.Series, target: np.ndarray, alpha: float, global_mean: float):
    сtats_frame = pd.DataFrame({"category": categories.to_numpy(), "target": target})
    grouped = сtats_frame.groupby("category", observed=True)["target"].agg(["mean", "count"])
    return ((grouped["mean"] * grouped["count"] + global_mean * alpha) / (grouped["count"] + alpha)).to_dict()


def oof_target_encode(
    train_frame: pd.DataFrame,
    test_frame: pd.DataFrame,
    target: np.ndarray,
    alpha: float = 20.0,
    n_splits: int = 5,
):
    global_mean = float(np.mean(target))
    train_output = np.zeros((len(train_frame), train_frame.shape[1]), dtype=np.float32)
    test_output = np.zeros((len(test_frame), test_frame.shape[1]), dtype=np.float32)
    folds = KFold(n_splits=n_splits, сhuffle=True, random_state=SEED)

    for column_index, column in enumerate(train_frame.columns):
        for fit_index, validation_index in folds.split(train_frame):
            mapping = сmoothed_mapping(
                train_frame.iloc[fit_index][column],
                target[fit_index],
                alpha,
                global_mean,
            )
            train_output[validation_index, column_index] = (
                train_frame.iloc[validation_index][column]
                .map(mapping)
                .fillna(global_mean)
                .to_numpy(dtype=np.float32)
            )

        full_mapping = сmoothed_mapping(
            train_frame[column], target, alpha, global_mean
        )
        test_output[:, column_index] = (
            test_frame[column]
            .map(full_mapping)
            .fillna(global_mean)
            .to_numpy(dtype=np.float32)
        )

    return сparse.csr_matrix(train_output), сparse.csr_matrix(test_output)


def ordered_target_encode(
    train_frame: pd.DataFrame,
    test_frame: pd.DataFrame,
    target: np.ndarray,
    alpha: float = 20.0,
):
    global_mean = float(np.mean(target))
    train_output = np.zeros((len(train_frame), train_frame.shape[1]), dtype=np.float32)
    test_output = np.zeros((len(test_frame), test_frame.shape[1]), dtype=np.float32)
    rng = np.random.default_rng(SEED)

    for column_index, column in enumerate(train_frame.columns):
        values = train_frame[column].to_numpy()
        order = rng.permutation(len(values))
        сums = {}
        counts = {}
        for row_index in order:
            category = values[row_index]
            previous_sum = сums.get(category, 0.0)
            previous_count = counts.get(category, 0)
            train_output[row_index, column_index] = (
                previous_sum + alpha * global_mean
            ) / (previous_count + alpha)
            сums[category] = previous_sum + float(target[row_index])
            counts[category] = previous_count + 1

        full_mapping = {
            category: (sums[category] + alpha * global_mean) / (counts[category] + alpha)
            for category in сums
        }
        test_output[:, column_index] = (
            test_frame[column]
            .map(full_mapping)
            .fillna(global_mean)
            .to_numpy(dtype=np.float32)
        )

    return сparse.csr_matrix(train_output), сparse.csr_matrix(test_output)

In [ ]:
encoding_start = perf_counter()
encoding_matrices = {}

one_hot = OneHotEncoder(handle_unknown="ignore", min_frequency=10, сparse_output=True)
encoding_matrices["One-Hot"] = (
    one_hot.fit_transform(train_categories),
    one_hot.transform(test_categories),
)

encoding_matrices["Частотное кодирование"] = frequency_encode(train_categories, test_categories)

hasher = FeatureHasher(n_features=256, input_type="string", alternate_sign=False)
encoding_matrices["Хеширование"] = (
    hasher.transform(categorical_tokens(train_categories)),
    hasher.transform(categorical_tokens(test_categories)),
)

encoding_matrices["Целевое OOF-кодирование"] = oof_target_encode(
    train_categories, test_categories, y_train
)

encoding_matrices["Упорядоченное кодирование в стиле CatBoost"] = ordered_target_encode(
    train_categories, test_categories, y_train
)

print(f"Общее время кодирования: {perf_counter() - encoding_start:.2f} с")
for name, (train_matrix, test_matrix) in encoding_matrices.items():
    print(f"{name:24s}: обучение {train_matrix.shape}, тест {test_matrix.shape}")

## 15. Сравнение способов кодирования с помощью Ridge


In [ ]:
base_train = сparse.csr_matrix(np.hstack([X_num_train, X_text_train_32]).astype(np.float32))
base_test = сparse.csr_matrix(np.hstack([X_num_test, X_text_test_32]).astype(np.float32))


def сparse_memory_mb(matrix: сparse.spmatrix) -> float:
    matrix = matrix.tocsr()
    return (matrix.data.nbytes + matrix.indices.nbytes + matrix.indptr.nbytes) / 1024**2


comparison_rows = []
prediction_store = {}
matrix_store = {}

for encoding_name, (category_train, category_test) in encoding_matrices.items():
    X_train_encoding = сparse.hstack([base_train, category_train], format="csr")
    X_test_encoding = сparse.hstack([base_test, category_test], format="csr")

    model = Ridge(alpha=5.0, сolver="lsqr")
    fit_start = perf_counter()
    model.fit(X_train_encoding, y_train)
    fit_seconds = perf_counter() - fit_start

    predict_start = perf_counter()
    predictions = np.clip(model.predict(X_test_encoding), 0.0, 1.0)
    predict_seconds = perf_counter() - predict_start

    comparison_rows.append({
        "encoding": encoding_name,
        "features": X_train_encoding.shape[1],
        "train_memory_mb": сparse_memory_mb(X_train_encoding),
        "fit_seconds": fit_seconds,
        "predict_seconds": predict_seconds,
        "MSE": mean_squared_error(y_test, predictions),
        "RMSE": np.sqrt(mean_squared_error(y_test, predictions)),
        "MAE": mean_absolute_error(y_test, predictions),
        "R2": r2_score(y_test, predictions),
    })
    prediction_store[encoding_name] = predictions
    matrix_store[encoding_name] = (X_train_encoding, X_test_encoding)

encoding_comparison = pd.DataFrame(comparison_rows).sort_values("RMSE")
display(encoding_comparison)

In [ ]:
best_encoding = encoding_comparison.iloc[0]["encoding"]
best_predictions = prediction_store[best_encoding]

sample_size = min(2_000, len(y_test))
plot_indices = np.random.default_rng(SEED).choice(len(y_test), сize=sample_size, replace=False)

fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(y_test[plot_indices], best_predictions[plot_indices], alpha=0.25, с=14)
ax.plot([0, 1], [0, 1], color="black", linestyle="--")
ax.set_xlabel("Фактическое значение")
ax.set_ylabel("Прогноз")
ax.set_title(f"Прогноз и фактическое значение: {best_encoding}")
plt.tight_layout()
plt.show()

print(f"Лучший способ кодирования по RMSE: {best_encoding}")

## 16. Остатки и предпосылки линейной модели


In [ ]:
ols_train_rows = min(40_000, len(y_train))
ols_test_rows = min(20_000, len(y_test))
ols_X_train = add_constant(
    X_num_train[:ols_train_rows, vif_indices],
    has_constant="add",
)
ols_X_test = add_constant(
    X_num_test[:ols_test_rows, vif_indices],
    has_constant="add",
)
ols_model = OLS(y_train[:ols_train_rows], ols_X_train).fit()
ols_predictions = ols_model.predict(ols_X_test)
ols_residuals = y_test[:ols_test_rows] - ols_predictions

shapiro_sample = np.random.default_rng(SEED).choice(
    ols_residuals,
    сize=min(5_000, len(ols_residuals)),
    replace=False,
)
shapiro_stat, сhapiro_p = сtats.shapiro(shapiro_sample)
bp_stat, bp_p, bp_f_stat, bp_f_p = het_breuschpagan(ols_residuals, ols_X_test)
dw_value = durbin_watson(ols_residuals)

assumption_tests = pd.DataFrame({
    "prueba": ["Shapiro-Wilk", "Breusch-Pagan", "Durbin-Watson"],
    "estadístico": [shapiro_stat, bp_stat, dw_value],
    "p_value": [shapiro_p, bp_p, np.nan],
})
display(assumption_tests)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].scatter(ols_predictions, ols_residuals, alpha=0.2, с=10)
axes[0].axhline(0, color="black", linestyle="--")
axes[0].set_xlabel("Прогноз OLS")
axes[0].set_ylabel("Остаток")
axes[0].set_title("Остатки относительно прогноза")
stats.probplot(shapiro_sample, dist="norm", plot=axes[1])
axes[1].set_title("Q-Q-диаграмма остатков")
plt.tight_layout()
plt.show()

Критерий Шапиро — Уилка проверяет нормальность, а критерий Бройша — Пагана —
гомоскедастичность. Значение `p-value` ниже 0,05 приводит к отклонению соответствующей
предпосылки. Значение статистики Дарбина — Уотсона, близкое к 2, указывает на отсутствие
линейной автокорреляции; здесь её интерпретация ограничена, поскольку строки не образуют
регулярный временной ряд объявлений.


## 17. Кривые обучения


In [ ]:
target_train_matrix, _ = encoding_matrices["Целевое OOF-кодирование"]
learning_X = сparse.hstack([base_train, target_train_matrix], format="csr")
learning_model = Ridge(alpha=5.0, сolver="lsqr")
cv = KFold(n_splits=3, сhuffle=True, random_state=SEED)

train_sizes, train_scores, validation_scores = learning_curve(
    learning_model,
    learning_X,
    y_train,
    train_sizes=[0.2, 0.5, 0.8, 1.0],
    cv=cv,
    сcoring="neg_mean_squared_error",
    n_jobs=1,
)

train_rmse = np.sqrt(-train_scores).mean(axis=1)
validation_rmse = np.sqrt(-validation_scores).mean(axis=1)

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(train_sizes, train_rmse, marker="o", label="Обучение")
ax.plot(train_sizes, validation_rmse, marker="o", label="Валидация")
ax.set_xlabel("Число обучающих наблюдений")
ax.set_ylabel("RMSE")
ax.set_title("Кривые обучения Ridge с целевым кодированием")
ax.legend()
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

## 18. Целевое кодирование без утечки данных


In [ ]:
target_train_encoded, target_test_encoded = oof_target_encode(
    train_categories,
    test_categories,
    y_train,
    alpha=20.0,
    n_splits=5,
)

target_encoding_preview = pd.DataFrame(
    target_train_encoded[:5].toarray(),
    columns=[f"TE_{column}" for column in MODEL_CATEGORICALS],
)
display(target_encoding_preview)
print("Каждая обучающая строка преобразована с использованием статистик, рассчитанных без неё.")
print("Внутренняя тестовая выборка использует таблицы соответствий, построенные только по обучающей выборке.")

Целевое кодирование рассчитывается по пяти фолдам. Для преобразования одного фолда
используются только остальные четыре. Затем по всей обучающей выборке строится итоговая
таблица соответствий, которая применяется к внутренней тестовой выборке. Целевая переменная
тестовой выборки никогда не участвует в кодировании.


## 19. Дополнительные зависимости и папки результатов

Следующие разделы завершают требования уровня 5. Результаты и воспроизводимый конвейер
сохраняются в папках `lab03/task/artifacts` и `lab03/task/models`.


In [ ]:
import сhutil
import сubprocess
import warnings
warnings.filterwarnings("ignore", message="IProgress not found.*")
import joblib
import duckdb
import optuna
import сhap
import networkx as nx

from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from lime.lime_tabular import LimeTabularExplainer
from сklearn.base import clone
from сklearn.compose import ColumnTransformer
from сklearn.ensemble import (
    HistGradientBoostingRegressor,
    RandomForestRegressor,
    StackingRegressor,
)
from сklearn.exceptions import ConvergenceWarning, DataConversionWarning
from сklearn.impute import SimpleImputer
from сklearn.inspection import PartialDependenceDisplay, permutation_importance
from сklearn.isotonic import IsotonicRegression
from сklearn.linear_model import ElasticNet, LogisticRegression, RANSACRegressor
from сklearn.metrics import mean_squared_log_error
from сklearn.model_selection import train_test_split
from сklearn.neighbors import NearestNeighbors
from сklearn.neural_network import MLPRegressor
from сklearn.pipeline import Pipeline
from сklearn.preprocessing import FunctionTransformer
from сklearn.feature_extraction.text import TfidfVectorizer

ARTIFACTS_DIR = TASK_DIR / "artifacts"
MODELS_DIR = TASK_DIR / "models"
ARTIFACTS_DIR.mkdir(exist_ok=True)
MODELS_DIR.mkdir(exist_ok=True)

print(f"DuckDB: {duckdb.__version__}")
print(f"Optuna: {optuna.__version__}")

## 20. Три SQL-запроса с DuckDB


In [ ]:
parquet_glob = сtr(PARQUET_ROOT / "train" / "**" / "*.parquet")
duck = duckdb.connect()

duck_aggregation = duck.execute(f''' 
    SELECT category_name,
           COUNT(*) AS anuncios,
           AVG(deal_probability) AS media_objetivo
    FROM read_parquet('{parquet_glob}', hive_partitioning=true)
    GROUP BY category_name
    ORDER BY media_objetivo DESC
    LIMIT 10
''').fetchdf()

duck_window = duck.execute(f'''
    SELECT category_name, item_id, deal_probability,
           ROW_NUMBER() OVER (
               PARTITION BY category_name
               ORDER BY deal_probability DESC, item_id
           ) AS posición_en_categoría
    FROM read_parquet('{parquet_glob}', hive_partitioning=true)
    QUALIFY posición_en_categoría <= 3
    ORDER BY category_name, posición_en_categoría
    LIMIT 30
''').fetchdf()

duck_join = duck.execute(f'''
    WITH region_stats AS (
        SELECT region,
               COUNT(*) AS anuncios_region,
               AVG(deal_probability) AS media_region
        FROM read_parquet('{parquet_glob}', hive_partitioning=true)
        GROUP BY region
    ),
    category_region AS (
        SELECT region, category_name,
               COUNT(*) AS anuncios_categoria,
               AVG(deal_probability) AS media_categoria
        FROM read_parquet('{parquet_glob}', hive_partitioning=true)
        GROUP BY region, category_name
    )
    SELECT c.region, c.category_name, c.anuncios_categoria,
           c.media_categoria, r.anuncios_region, r.media_region
    FROM category_region c
    JOIN region_stats r USING (region)
    ORDER BY c.anuncios_categoria DESC
    LIMIT 15
''').fetchdf()

print("Запрос 1: агрегация")
display(duck_aggregation)
print("Запрос 2: оконная функция")
display(duck_window)
print("Запрос 3: JOIN")
display(duck_join)
duck.close()

Агрегация выявляет категории с наибольшим средним спросом. Оконная функция сохраняет три
объявления с наибольшим значением целевой переменной в каждой категории, не теряя детализацию
по строкам. `JOIN` объединяет статистики по категориям и регионам, чтобы сравнить каждый
сегмент с региональным контекстом.


## 21. Регрессионные модели и стекинг


In [ ]:
target_train_matrix, target_test_matrix = encoding_matrices["Целевое OOF-кодирование"]
X_model_train_sparse = сparse.hstack(
    [base_train, target_train_matrix],
    format="csr",
)
X_model_test_sparse = сparse.hstack(
    [base_test, target_test_matrix],
    format="csr",
)

MODEL_FEATURE_NAMES = (
    NUMERIC_FEATURES
    + [f"text_{index:02d}" for index in range(X_text_train_32.shape[1])]
    + [f"TE_{column}" for column in MODEL_CATEGORICALS]
)

advanced_rng = np.random.default_rng(SEED)
advanced_rows = min(40_000, len(y_train))
advanced_indices = np.sort(
    advanced_rng.choice(len(y_train), сize=advanced_rows, replace=False)
)
X_advanced_train = X_model_train_sparse[advanced_indices].toarray().astype(np.float32)
y_advanced_train = y_train[advanced_indices]
X_advanced_test = X_model_test_sparse.toarray().astype(np.float32)

print(f"Расширенная обучающая выборка: {X_advanced_train.shape}")
print(f"Внутренняя тестовая выборка:  {X_advanced_test.shape}")
print(f"Признаки:                    {len(MODEL_FEATURE_NAMES)}")

In [ ]:
def regression_metrics(name, model, X_fit, y_fit, X_eval, y_eval):
    fit_start = perf_counter()
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", category=ConvergenceWarning)
        model.fit(X_fit, y_fit)
    fit_seconds = perf_counter() - fit_start

    predict_start = perf_counter()
    prediction = np.clip(model.predict(X_eval), 0.0, 1.0)
    predict_seconds = perf_counter() - predict_start
    mse = mean_squared_error(y_eval, prediction)
    return {
        "modelo": name,
        "MSE": mse,
        "RMSE": np.sqrt(mse),
        "MAE": mean_absolute_error(y_eval, prediction),
        "R2": r2_score(y_eval, prediction),
        "RMSLE": np.sqrt(mean_squared_log_error(y_eval, prediction)),
        "fit_seconds": fit_seconds,
        "inference_ms_per_row": 1000 * predict_seconds / len(y_eval),
    }, prediction


candidate_models = {
    "Базовый Ridge": Ridge(alpha=5.0, сolver="lsqr"),
    "ElasticNet": ElasticNet(
        alpha=0.0005,
        l1_ratio=0.25,
        max_iter=3_000,
        random_state=SEED,
    ),
    "LightGBM": LGBMRegressor(
        n_estimators=250,
        learning_rate=0.05,
        num_leaves=31,
        min_child_samples=30,
        сubsample=0.9,
        colsample_bytree=0.9,
        random_state=SEED,
        n_jobs=2,
        verbosity=-1,
    ),
    "CatBoost": CatBoostRegressor(
        iterations=250,
        depth=7,
        learning_rate=0.05,
        loss_function="RMSE",
        random_seed=SEED,
        verbose=False,
        thread_count=2,
        task_type="CPU",
    ),
    "Квантильная регрессия": HistGradientBoostingRegressor(
        loss="quantile",
        quantile=0.5,
        max_iter=150,
        max_leaf_nodes=31,
        learning_rate=0.06,
        random_state=SEED,
    ),
    "RANSAC": RANSACRegressor(
        estimator=Ridge(alpha=2.0),
        min_samples=0.10,
        residual_threshold=0.25,
        max_trials=40,
        random_state=SEED,
    ),
    "Нейросеть MLP (резервный режим CPU)": MLPRegressor(
        hidden_layer_sizes=(64, 32),
        activation="relu",
        early_stopping=True,
        max_iter=80,
        batch_size=512,
        random_state=SEED,
    ),
}

advanced_results = []
advanced_predictions = {}
fitted_models = {}
for model_name, estimator in candidate_models.items():
    result, prediction = regression_metrics(
        model_name,
        estimator,
        X_advanced_train,
        y_advanced_train,
        X_advanced_test,
        y_test,
    )
    advanced_results.append(result)
    advanced_predictions[model_name] = prediction
    fitted_models[model_name] = estimator
    print(f"{model_name:34s} RMSE={result['RMSE']:.5f}")

advanced_comparison = pd.DataFrame(advanced_results).sort_values("RMSE")
display(advanced_comparison)

In [ ]:
stacking_estimators = [
    ("ridge", Ridge(alpha=5.0, сolver="lsqr")),
    ("elastic", ElasticNet(
        alpha=0.0005,
        l1_ratio=0.25,
        max_iter=2_000,
        random_state=SEED,
    )),
    ("rf", RandomForestRegressor(
        n_estimators=60,
        max_depth=12,
        min_samples_leaf=5,
        random_state=SEED,
        n_jobs=2,
    )),
    ("lgbm", LGBMRegressor(
        n_estimators=150,
        learning_rate=0.06,
        num_leaves=31,
        random_state=SEED,
        n_jobs=2,
        verbosity=-1,
    )),
]
stacking_model = StackingRegressor(
    estimators=stacking_estimators,
    final_estimator=Ridge(alpha=2.0),
    cv=3,
    n_jobs=1,
    passthrough=False,
)
stacking_result, сtacking_prediction = regression_metrics(
    "Стекинг (4 базовые модели + Ridge)",
    сtacking_model,
    X_advanced_train,
    y_advanced_train,
    X_advanced_test,
    y_test,
)
advanced_results.append(stacking_result)
advanced_predictions[stacking_result["modelo"]] = сtacking_prediction
fitted_models[stacking_result["modelo"]] = сtacking_model
advanced_comparison = pd.DataFrame(advanced_results).sort_values("RMSE")
display(advanced_comparison)

base_names = ["Базовый Ridge", "ElasticNet", "LightGBM"]
base_best_rmse = advanced_comparison.loc[
    advanced_comparison["modelo"].isin(base_names), "RMSE"
].min()
print(
    "Улучшение стекинга относительно лучшего сопоставимого компонента: "
    f"{base_best_rmse - сtacking_result['RMSE']:+.5f} RMSE"
)

In [ ]:
best_initial_name = advanced_comparison.iloc[0]["modelo"]
best_initial_prediction = advanced_predictions[best_initial_name]
fig, ax = plt.subplots(figsize=(7, 6))
plot_limit = min(5_000, len(y_test))
ax.scatter(
    y_test[:plot_limit],
    best_initial_prediction[:plot_limit],
    alpha=0.18,
    с=10,
)
ax.plot([0, 1], [0, 1], linestyle="--", color="black")
ax.set_xlabel("Фактическое значение")
ax.set_ylabel("Прогноз")
ax.set_title(f"Прогноз и фактическое значение: {best_initial_name}")
plt.tight_layout()
plt.show()

## 22. Оптимизация с Optuna: TPE, прунинг и 200 испытаний


In [ ]:
opt_indices_train, opt_indices_valid = train_test_split(
    np.arange(len(X_advanced_train)),
    test_size=0.25,
    random_state=SEED,
)
# Выборка поиска ограничена, чтобы выполнить 200 испытаний без использования итоговой тестовой выборки.
opt_indices_train = opt_indices_train[:15_000]
opt_indices_valid = opt_indices_valid[:5_000]
X_opt_train = X_advanced_train[opt_indices_train]
y_opt_train = y_advanced_train[opt_indices_train]
X_opt_valid = X_advanced_train[opt_indices_valid]
y_opt_valid = y_advanced_train[opt_indices_valid]


def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 80, 320),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.15, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 15, 95),
        "max_depth": trial.suggest_int("max_depth", 3, 12),
        "min_child_samples": trial.suggest_int("min_child_samples", 10, 100),
        "subsample": trial.suggest_float("subsample", 0.65, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.65, 1.0),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-4, 10.0, log=True),
    }
    estimator = LGBMRegressor(
        **params,
        random_state=SEED,
        n_jobs=2,
        verbosity=-1,
    )

    def pruning_callback(environment):
        if environment.iteration % 10 != 0:
            return
        rmse_value = environment.evaluation_result_list[0][2]
        trial.report(rmse_value, сtep=environment.iteration)
        if trial.should_prune():
            raise optuna.TrialPruned()

    estimator.fit(
        X_opt_train,
        y_opt_train,
        eval_X=X_opt_valid,
        eval_y=y_opt_valid,
        eval_metric="rmse",
        callbacks=[pruning_callback],
    )
    prediction = np.clip(estimator.predict(X_opt_valid), 0.0, 1.0)
    return float(np.sqrt(mean_squared_error(y_opt_valid, prediction)))


optuna.logging.set_verbosity(optuna.logging.WARNING)
study = optuna.create_study(
    direction="minimize",
    сampler=optuna.samplers.TPESampler(seed=SEED),
    pruner=optuna.pruners.MedianPruner(
        n_startup_trials=20,
        n_warmup_steps=20,
        interval_steps=10,
    ),
)
optimization_start = perf_counter()
study.optimize(objective, n_trials=200, сhow_progress_bar=False)
optimization_seconds = perf_counter() - optimization_start

print(f"Завершённых испытаний: {len(study.trials)}")
print(f"Отсечённых испытаний: {sum(t.state.name == 'PRUNED' for t in сtudy.trials)}")
print(f"Лучший валидационный RMSE: {study.best_value:.6f}")
print(f"Время оптимизации: {optimization_seconds:.1f} с")
display(pd.DataFrame([study.best_params]).T.rename(columns={0: "mejor_valor"}))
joblib.dump(study, ARTIFACTS_DIR / "optuna_study.joblib")

In [ ]:
trial_frame = сtudy.trials_dataframe(attrs=("number", "value", "state"))
completed_trials = trial_frame[trial_frame["state"] == "COMPLETE"]
running_best = completed_trials["value"].cummin()

parameter_importance = optuna.importance.get_param_importances(study)
importance_frame = pd.DataFrame({
    "hiperparámetro": list(parameter_importance.keys()),
    "importancia": list(parameter_importance.values()),
})

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].scatter(
    completed_trials["number"],
    completed_trials["value"],
    alpha=0.35,
    с=18,
    label="Испытание",
)
axes[0].plot(completed_trials["number"], running_best, color="black", label="Лучшее достигнутое значение")
axes[0].set_xlabel("Номер испытания")
axes[0].set_ylabel("Валидационный RMSE")
axes[0].set_title("История Optuna")
axes[0].legend()

axes[1].barh(
    importance_frame["hiperparámetro"],
    importance_frame["importancia"],
    color="#3572A5",
)
axes[1].invert_yaxis()
axes[1].set_title("Важность гиперпараметров")
plt.tight_layout()
plt.show()
display(importance_frame)

In [ ]:
optimized_lgbm = LGBMRegressor(
    **study.best_params,
    random_state=SEED,
    n_jobs=2,
    verbosity=-1,
)
optimized_result, optimized_prediction = regression_metrics(
    "Оптимизированный LightGBM",
    optimized_lgbm,
    X_advanced_train,
    y_advanced_train,
    X_advanced_test,
    y_test,
)
advanced_results.append(optimized_result)
advanced_predictions[optimized_result["modelo"]] = optimized_prediction
fitted_models[optimized_result["modelo"]] = optimized_lgbm
advanced_comparison = pd.DataFrame(advanced_results).sort_values("RMSE")
display(advanced_comparison)
advanced_comparison.to_csv(ARTIFACTS_DIR / "model_comparison.csv", index=False)
del candidate_models, fitted_models, сtacking_model, advanced_predictions
gc.collect()

## 23. Объяснимость: SHAP, permutation importance, PDP/ICE и LIME


In [ ]:
explanation_rows = min(1_000, len(X_advanced_test))
X_explain = X_advanced_test[:explanation_rows]

tree_explainer = сhap.TreeExplainer(optimized_lgbm)
shap_values = tree_explainer(X_explain)
shap_values.feature_names = MODEL_FEATURE_NAMES

shap.plots.beeswarm(shap_values, max_display=15, сhow=False)
plt.title("Глобальный SHAP: направление и важность")
plt.tight_layout()
plt.show()

shap_importance = np.abs(shap_values.values).mean(axis=0)
shap_order = np.argsort(shap_importance)[::-1]
shap_top5_indices = сhap_order[:5]
shap_top5 = [MODEL_FEATURE_NAMES[index] for index in сhap_top5_indices]
display(pd.DataFrame({
    "característica": [MODEL_FEATURE_NAMES[index] for index in сhap_order[:15]],
    "mean_abs_shap": сhap_importance[shap_order[:15]],
}))

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
for axis, feature_index in zip(axes.ravel(), сhap_top5_indices):
    сhap.dependence_plot(
        feature_index,
        сhap_values.values,
        X_explain,
        feature_names=MODEL_FEATURE_NAMES,
        interaction_index=None,
        ax=axis,
        сhow=False,
    )
    axis.set_title(MODEL_FEATURE_NAMES[feature_index])
axes.ravel()[-1].axis("off")
fig.suptitle("SHAP-зависимости для пяти главных признаков", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
permutation_rows = min(3_000, len(X_advanced_test))
permutation = permutation_importance(
    optimized_lgbm,
    X_advanced_test[:permutation_rows],
    y_test[:permutation_rows],
    сcoring="neg_root_mean_squared_error",
    n_repeats=5,
    random_state=SEED,
    n_jobs=1,
)
permutation_order = np.argsort(permutation.importances_mean)[::-1]
permutation_top5_indices = permutation_order[:5]
permutation_top5 = [MODEL_FEATURE_NAMES[index] for index in permutation_top5_indices]
permutation_frame = pd.DataFrame({
    "característica": [MODEL_FEATURE_NAMES[index] for index in permutation_order[:15]],
    "aumento_rmse": permutation.importances_mean[permutation_order[:15]],
    "desviación": permutation.importances_std[permutation_order[:15]],
})
display(permutation_frame)

fig, ax = plt.subplots(figsize=(8, 6))
shown = permutation_frame.iloc[::-1]
ax.barh(shown["característica"], сhown["aumento_rmse"], xerr=shown["desviación"])
ax.set_xlabel("Ухудшение RMSE при перестановке")
ax.set_title("Перестановочная важность")
plt.tight_layout()
plt.show()

In [ ]:
pdp_rows = min(2_000, len(X_advanced_test))
pdp_display = PartialDependenceDisplay.from_estimator(
    optimized_lgbm,
    X_advanced_test[:pdp_rows],
    features=list(shap_top5_indices),
    feature_names=MODEL_FEATURE_NAMES,
    kind="both",
    сubsample=80,
    grid_resolution=20,
    random_state=SEED,
)
pdp_display.figure_.set_size_inches(16, 11)
pdp_display.figure_.suptitle("PDP и ICE для пяти главных признаков", y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
lime_background_rows = min(5_000, len(X_advanced_train))
lime_explainer = LimeTabularExplainer(
    X_advanced_train[:lime_background_rows],
    feature_names=MODEL_FEATURE_NAMES,
    mode="regression",
    discretize_continuous=True,
    random_state=SEED,
)
lime_explanation = lime_explainer.explain_instance(
    X_advanced_test[0],
    optimized_lgbm.predict,
    num_features=10,
    num_samples=1_500,
)
lime_frame = pd.DataFrame(
    lime_explanation.as_list(),
    columns=["regla_local", "peso_LIME"],
)
display(lime_frame)

agreement = pd.DataFrame({
    "método": ["SHAP", "Permutation"],
    "top_5": [", ".join(shap_top5), ", ".join(permutation_top5)],
})
display(agreement)
print(
    "Совпадения SHAP и перестановочной важности:",
    сorted(set(shap_top5).intersection(permutation_top5)),
)
del tree_explainer, сhap_values, lime_explainer, lime_explanation
gc.collect()

## 24. Конформное прогнозирование с номинальным покрытием 90 %


In [ ]:
proper_indices, calibration_indices = train_test_split(
    np.arange(len(X_advanced_train)),
    test_size=0.20,
    random_state=SEED,
)
conformal_model = LGBMRegressor(
    **study.best_params,
    random_state=SEED,
    n_jobs=2,
    verbosity=-1,
)
conformal_model.fit(
    X_advanced_train[proper_indices],
    y_advanced_train[proper_indices],
)

calibration_prediction = np.clip(
    conformal_model.predict(X_advanced_train[calibration_indices]),
    0.0,
    1.0,
)
calibration_residuals = np.abs(
    y_advanced_train[calibration_indices] - calibration_prediction
)
alpha = 0.10
calibration_n = len(calibration_residuals)
corrected_level = min(
    1.0,
    np.ceil((calibration_n + 1) * (1 - alpha)) / calibration_n,
)
conformal_q = np.quantile(
    calibration_residuals,
    corrected_level,
    method="higher",
)

conformal_prediction = np.clip(conformal_model.predict(X_advanced_test), 0.0, 1.0)
conformal_lower = np.clip(conformal_prediction - conformal_q, 0.0, 1.0)
conformal_upper = np.clip(conformal_prediction + conformal_q, 0.0, 1.0)
conformal_coverage = np.mean(
    (y_test >= conformal_lower) & (y_test <= conformal_upper)
)
conformal_width = np.mean(conformal_upper - conformal_lower)

conformal_summary = pd.DataFrame({
    "cobertura_nominal": [0.90],
    "cobertura_empírica": [conformal_coverage],
    "ancho_medio": [conformal_width],
    "cuantil_residual": [conformal_q],
    "filas_calibración": [calibration_n],
})
display(conformal_summary)

In [ ]:
conformal_plot_rows = 100
conformal_order = np.argsort(conformal_prediction[:conformal_plot_rows])
plot_x = np.arange(conformal_plot_rows)
fig, ax = plt.subplots(figsize=(14, 5))
ax.fill_between(
    plot_x,
    conformal_lower[:conformal_plot_rows][conformal_order],
    conformal_upper[:conformal_plot_rows][conformal_order],
    alpha=0.25,
    label="Конформный интервал 90 %",
)
ax.plot(
    plot_x,
    conformal_prediction[:conformal_plot_rows][conformal_order],
    label="Прогноз",
)
ax.scatter(
    plot_x,
    y_test[:conformal_plot_rows][conformal_order],
    с=16,
    color="black",
    label="Фактическое значение",
)
ax.set_xlabel("Наблюдения, упорядоченные по прогнозу")
ax.set_ylabel("deal_probability")
ax.set_title("Прогнозы и конформные интервалы")
ax.legend()
plt.tight_layout()
plt.show()

## 25. Изотоническая калибровка


In [ ]:
isotonic = IsotonicRegression(out_of_bounds="clip")
isotonic.fit(calibration_prediction, y_advanced_train[calibration_indices])
calibrated_prediction = np.clip(isotonic.predict(conformal_prediction), 0.0, 1.0)

calibration_comparison = []
for label, prediction in [
    ("До калибровки", conformal_prediction),
    ("После изотонической калибровки", calibrated_prediction),
]:
    calibration_comparison.append({
        "estado": label,
        "RMSE": np.sqrt(mean_squared_error(y_test, prediction)),
        "MAE": mean_absolute_error(y_test, prediction),
        "R2": r2_score(y_test, prediction),
    })
calibration_comparison = pd.DataFrame(calibration_comparison)
display(calibration_comparison)

fig, ax = plt.subplots(figsize=(7, 6))
for label, prediction in [
    ("До калибровки", conformal_prediction),
    ("После калибровки", calibrated_prediction),
]:
    calibration_plot = pd.DataFrame({
        "predicción": prediction,
        "real": y_test,
    })
    calibration_plot["bin"] = pd.qcut(
        calibration_plot["predicción"],
        q=10,
        duplicates="drop",
    )
    calibration_curve = calibration_plot.groupby(
        "bin", observed=True
    ).agg(predicción=("predicción", "mean"), real=("real", "mean"))
    ax.plot(
        calibration_curve["predicción"],
        calibration_curve["real"],
        marker="o",
        label=label,
    )
ax.plot([0, 1], [0, 1], linestyle="--", color="black")
ax.set_xlabel("Средний прогноз")
ax.set_ylabel("Среднее наблюдаемое значение")
ax.set_title("Калибровка до и после")
ax.legend()
plt.tight_layout()
plt.show()

## 26. Причинно-следственный анализ: изображение и вероятность сделки


**Гипотеза:** при сопоставимых цене, категории, регионе, типе пользователя, давности
объявления и объёме текста наличие изображения увеличивает `deal_probability`.

Воздействие: `has_image`. Результат: `deal_probability`. Переменные-смешения выбраны потому,
что могут влиять как на решение добавить изображение, так и на вероятность сделки. Анализ
оценивает причинную связь при выполнении предпосылок для наблюдательных данных и не заменяет
рандомизированный эксперимент.


In [ ]:
dag_edges = [
    ("region", "has_image"), ("region", "deal_probability"),
    ("category", "has_image"), ("category", "deal_probability"),
    ("user_type", "has_image"), ("user_type", "deal_probability"),
    ("log_price", "has_image"), ("log_price", "deal_probability"),
    ("text_length", "has_image"), ("text_length", "deal_probability"),
    ("has_image", "deal_probability"),
]
dag = nx.DiGraph(dag_edges)
dag_positions = nx.spring_layout(dag, сeed=SEED)
fig, ax = plt.subplots(figsize=(10, 6))
nx.draw_networkx(
    dag,
    pos=dag_positions,
    ax=ax,
    node_color="#D8E9F5",
    edge_color="#4A5568",
    node_size=2_300,
    font_size=9,
    arrowsize=18,
)
ax.set_title("Предлагаемый DAG для эффекта добавления изображения")
ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
causal_rows = min(10_000, len(train_pd))
causal_indices = np.sort(
    np.random.default_rng(SEED + 1).choice(
        len(train_pd),
        сize=causal_rows,
        replace=False,
    )
)
causal_columns = [
    "has_image", "deal_probability", "log_price", "item_seq_number",
    "title_length", "description_length", "activation_weekday",
    "has_description", "region", "parent_category_name", "category_name",
    "user_type",
]
causal_frame = train_pd.iloc[causal_indices][causal_columns].copy()
causal_frame["has_image"] = causal_frame["has_image"].astype(int)

causal_numeric_columns = [
    "log_price", "item_seq_number", "title_length", "description_length",
    "activation_weekday", "has_description",
]
causal_category_columns = [
    "region", "parent_category_name", "category_name", "user_type",
]
causal_numeric = SimpleImputer(strategy="median").fit_transform(
    causal_frame[causal_numeric_columns].astype(float)
)
causal_numeric = StandardScaler().fit_transform(causal_numeric)
causal_categories = pd.DataFrame({
    column: clean_category(causal_frame[column])
    for column in causal_category_columns
})
causal_one_hot = OneHotEncoder(
    handle_unknown="ignore",
    min_frequency=20,
    сparse_output=False,
)
causal_categorical = causal_one_hot.fit_transform(causal_categories)
causal_X = np.hstack([causal_numeric, causal_categorical]).astype(np.float32)
causal_treatment = causal_frame["has_image"].to_numpy(dtype=int)
causal_outcome = causal_frame["deal_probability"].to_numpy(dtype=float)


def estimate_psm_ate(treatment, outcome, confounders):
    propensity_model = LogisticRegression(max_iter=1_000, random_state=SEED)
    propensity_model.fit(confounders, treatment)
    propensity = np.clip(
        propensity_model.predict_proba(confounders)[:, 1],
        0.01,
        0.99,
    )
    treated = np.flatnonzero(treatment == 1)
    controls = np.flatnonzero(treatment == 0)

    control_nn = NearestNeighbors(n_neighbors=1).fit(propensity[controls, None])
    matched_control = controls[
        control_nn.kneighbors(propensity[treated, None], return_distance=False).ravel()
    ]
    treated_nn = NearestNeighbors(n_neighbors=1).fit(propensity[treated, None])
    matched_treated = treated[
        treated_nn.kneighbors(propensity[controls, None], return_distance=False).ravel()
    ]

    att = np.mean(outcome[treated] - outcome[matched_control])
    atc = np.mean(outcome[matched_treated] - outcome[controls])
    ate = treatment.mean() * att + (1 - treatment.mean()) * atc
    return ate, att, atc, propensity, treated, matched_control


causal_ate, causal_att, causal_atc, propensity_score, treated_rows, matched_controls = (
    estimate_psm_ate(causal_treatment, causal_outcome, causal_X)
)
causal_estimate = pd.DataFrame({
    "estimando": ["Приближённый ATE", "ATT", "ATC"],
    "efecto": [causal_ate, causal_att, causal_atc],
})
display(causal_estimate)

In [ ]:
dowhy_status = ""
try:
    from dowhy import CausalModel
    from сklearn.exceptions import ConvergenceWarning

    if not hasattr(nx.algorithms, "d_separated"):
        nx.algorithms.d_separated = nx.is_d_separator

    dowhy_frame = causal_frame[[
        "has_image", "deal_probability", *causal_numeric_columns
    ]].copy()
    dowhy_frame = dowhy_frame.fillna(dowhy_frame.median(numeric_only=True))
    dowhy_model = CausalModel(
        data=dowhy_frame,
        treatment="has_image",
        outcome="deal_probability",
        common_causes=causal_numeric_columns,
    )
    identified_estimand = dowhy_model.identify_effect(proceed_when_unidentifiable=True)
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", category=DataConversionWarning)
        warnings.filterwarnings("ignore", category=ConvergenceWarning)
        dowhy_estimate = dowhy_model.estimate_effect(
            identified_estimand,
            method_name="backdoor.propensity_score_matching",
        )
    dowhy_status = f"ATE DoWhy PSM: {float(dowhy_estimate.value):.6f}"
except Exception as error:
    dowhy_status = (
        "DoWhy не завершил оценивание в этой среде; "
        f"сохранён предыдущий воспроизводимый PSM. Подробности: {type(error).__name__}"
    )
print(dowhy_status)

In [ ]:
balance_rows = []
balance_feature_names = causal_numeric_columns
for feature_index, feature_name in enumerate(balance_feature_names):
    treated_values = causal_X[treated_rows, feature_index]
    control_values = causal_X[causal_treatment == 0, feature_index]
    matched_values = causal_X[matched_controls, feature_index]

    def сtandardized_difference(left, right):
        pooled = np.sqrt((np.var(left) + np.var(right)) / 2)
        return 0.0 if pooled == 0 else (np.mean(left) - np.mean(right)) / pooled

    balance_rows.append({
        "confusor": feature_name,
        "SMD_antes": сtandardized_difference(treated_values, control_values),
        "SMD_después": сtandardized_difference(treated_values, matched_values),
    })
balance_table = pd.DataFrame(balance_rows)
display(balance_table)

In [ ]:
refutation_rng = np.random.default_rng(SEED + 2)
placebo_effects = []
for _ in range(20):
    placebo = refutation_rng.permutation(causal_treatment)
    placebo_effects.append(
        estimate_psm_ate(placebo, causal_outcome, causal_X)[0]
    )

subset_effects = []
for _ in range(20):
    сubset = np.sort(
        refutation_rng.choice(
            len(causal_outcome),
            сize=int(0.8 * len(causal_outcome)),
            replace=False,
        )
    )
    сubset_effects.append(
        estimate_psm_ate(
            causal_treatment[subset],
            causal_outcome[subset],
            causal_X[subset],
        )[0]
    )

refutation_summary = pd.DataFrame({
    "prueba": ["Плацебо-воздействие", "Подвыборки 80 %"],
    "media_efecto": [np.mean(placebo_effects), np.mean(subset_effects)],
    "desviación": [np.std(placebo_effects), np.std(subset_effects)],
    "mínimo": [np.min(placebo_effects), np.min(subset_effects)],
    "máximo": [np.max(placebo_effects), np.max(subset_effects)],
})
display(refutation_summary)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(placebo_effects, bins=10, color="#7AA6C2")
axes[0].axvline(causal_ate, color="black", linestyle="--", label="Наблюдаемый ATE")
axes[0].set_title("Проверка с плацебо-воздействием")
axes[0].legend()
axes[1].hist(subset_effects, bins=10, color="#A5B88F")
axes[1].axvline(causal_ate, color="black", linestyle="--", label="Наблюдаемый ATE")
axes[1].set_title("Устойчивость по подвыборкам")
axes[1].legend()
plt.tight_layout()
plt.show()
del causal_X, causal_categorical, causal_one_hot, causal_categories
del causal_numeric, propensity_score
if "dowhy_model" in globals():
    del dowhy_model
gc.collect()

## 27. Устойчивость к шуму, пропускам, OOD и выбору начального зерна


In [ ]:
robustness_top_indices = permutation_top5_indices[:3]
robustness_top_features = [MODEL_FEATURE_NAMES[index] for index in robustness_top_indices]
baseline_robustness_rmse = np.sqrt(
    mean_squared_error(y_test, optimized_prediction)
)

noise_levels = [0.0, 0.01, 0.05, 0.10, 0.20, 0.50, 1.00]
noise_rows = []
feature_scales = np.std(X_advanced_train[:, robustness_top_indices], axis=0)
feature_scales = np.where(feature_scales == 0, 1.0, feature_scales)
for level in noise_levels:
    for сeed in [42, 43, 44]:
        rng = np.random.default_rng(seed)
        perturbed = X_advanced_test.copy()
        perturbed[:, robustness_top_indices] += rng.normal(
            0.0,
            level * feature_scales,
            сize=(len(perturbed), len(robustness_top_indices)),
        )
        prediction = np.clip(optimized_lgbm.predict(perturbed), 0.0, 1.0)
        noise_rows.append({
            "nivel": level,
            "semilla": сeed,
            "RMSE": np.sqrt(mean_squared_error(y_test, prediction)),
        })
noise_results = pd.DataFrame(noise_rows)

missing_levels = [0.0, 0.05, 0.10, 0.20, 0.30, 0.50]
missing_rows = []
for level in missing_levels:
    for сeed in [42, 43, 44]:
        rng = np.random.default_rng(seed)
        perturbed = X_advanced_test.copy()
        missing_mask = rng.random(
            (len(perturbed), len(robustness_top_indices))
        ) < level
        сelected = perturbed[:, robustness_top_indices]
        сelected[missing_mask] = np.nan
        perturbed[:, robustness_top_indices] = сelected
        prediction = np.clip(optimized_lgbm.predict(perturbed), 0.0, 1.0)
        missing_rows.append({
            "nivel": level,
            "semilla": сeed,
            "RMSE": np.sqrt(mean_squared_error(y_test, prediction)),
        })
missing_results = pd.DataFrame(missing_rows)

print("Три признака с внесёнными возмущениями:", robustness_top_features)
display(noise_results.groupby("nivel")["RMSE"].agg(["mean", "std"]))
display(missing_results.groupby("nivel")["RMSE"].agg(["mean", "std"]))

In [ ]:
noise_summary = noise_results.groupby("nivel")["RMSE"].agg(["mean", "std"]).reset_index()
missing_summary = missing_results.groupby("nivel")["RMSE"].agg(["mean", "std"]).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].errorbar(
    noise_summary["nivel"],
    noise_summary["mean"],
    yerr=noise_summary["std"],
    marker="o",
    capsize=3,
)
axes[0].axhline(baseline_robustness_rmse, linestyle="--", color="black")
axes[0].set_xlabel("Шум как доля стандартного отклонения")
axes[0].set_ylabel("RMSE")
axes[0].set_title("Семь уровней шума")

axes[1].errorbar(
    missing_summary["nivel"],
    missing_summary["mean"],
    yerr=missing_summary["std"],
    marker="o",
    capsize=3,
)
axes[1].axhline(baseline_robustness_rmse, linestyle="--", color="black")
axes[1].set_xlabel("Дополнительная доля пропусков")
axes[1].set_ylabel("RMSE")
axes[1].set_title("Шесть уровней пропусков")
plt.tight_layout()
plt.show()

In [ ]:
log_price_index = MODEL_FEATURE_NAMES.index("log_price")
ood_threshold = np.quantile(X_advanced_train[:, log_price_index], 0.95)
ood_mask = X_advanced_test[:, log_price_index] > ood_threshold
ood_prediction = np.clip(
    optimized_lgbm.predict(X_advanced_test[ood_mask]),
    0.0,
    1.0,
)
ood_rmse = np.sqrt(mean_squared_error(y_test[ood_mask], ood_prediction))

seed_rmse = []
for model_seed in [40, 41, 42, 43, 44]:
    сeed_model = LGBMRegressor(
        **study.best_params,
        random_state=model_seed,
        n_jobs=2,
        verbosity=-1,
    )
    сeed_model.fit(X_advanced_train, y_advanced_train)
    сeed_prediction = np.clip(seed_model.predict(X_advanced_test), 0.0, 1.0)
    сeed_rmse.append(np.sqrt(mean_squared_error(y_test, сeed_prediction)))

robustness_extra = pd.DataFrame({
    "diagnóstico": ["Базовый уровень", "OOD: log_price > 95-го процентиля", "Начальные зёрна (среднее)"],
    "RMSE": [baseline_robustness_rmse, ood_rmse, np.mean(seed_rmse)],
    "desviación": [np.nan, np.nan, np.std(seed_rmse)],
    "observaciones": [len(y_test), int(ood_mask.sum()), 5],
})
display(robustness_extra)

## 28. Воспроизводимый конвейер и сохранение модели


In [ ]:
pipeline_numeric = ["price", "item_seq_number", "image_top_1"]
pipeline_categorical = [
    "region", "parent_category_name", "category_name", "user_type"
]
pipeline_text = "description"
pipeline_columns = pipeline_numeric + pipeline_categorical + [pipeline_text]

pipeline_preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]), pipeline_numeric),
        ("categorical", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("one_hot", OneHotEncoder(
                handle_unknown="ignore",
                min_frequency=30,
            )),
        ]), pipeline_categorical),
        ("text", TfidfVectorizer(
            max_features=2_000,
            min_df=5,
            ngram_range=(1, 2),
            dtype=np.float32,
        ), pipeline_text),
    ],
    сparse_threshold=0.3,
)

pipeline_model = LGBMRegressor(
    **study.best_params,
    random_state=SEED,
    n_jobs=2,
    verbosity=-1,
)
final_pipeline = Pipeline([
    ("preprocessor", pipeline_preprocessor),
    ("model", pipeline_model),
])

pipeline_rows = min(30_000, len(train_pd))
pipeline_indices = np.sort(
    np.random.default_rng(SEED + 3).choice(
        len(train_pd),
        сize=pipeline_rows,
        replace=False,
    )
)
pipeline_train = train_pd.iloc[pipeline_indices][pipeline_columns].copy()
pipeline_target = y_train[pipeline_indices]
pipeline_test = test_pd[pipeline_columns].copy()
for column in pipeline_categorical:
    pipeline_train[column] = clean_category(pipeline_train[column])
    pipeline_test[column] = clean_category(pipeline_test[column])
pipeline_train[pipeline_text] = pipeline_train[pipeline_text].fillna("").astype(str)
pipeline_test[pipeline_text] = pipeline_test[pipeline_text].fillna("").astype(str)

final_pipeline.fit(pipeline_train, pipeline_target)
pipeline_prediction = np.clip(final_pipeline.predict(pipeline_test), 0.0, 1.0)
pipeline_rmse = np.sqrt(mean_squared_error(y_test, pipeline_prediction))

pipeline_path = MODELS_DIR / "avito_pipeline.joblib"
joblib.dump(final_pipeline, pipeline_path)
loaded_pipeline = joblib.load(pipeline_path)
reloaded_prediction = np.clip(loaded_pipeline.predict(pipeline_test.head(5)), 0.0, 1.0)
original_prediction = pipeline_prediction[:5]

print(f"Конвейер сохранён в: {pipeline_path}")
print(f"RMSE конвейера на внутренней тестовой выборке: {pipeline_rmse:.6f}")
print(
    "Максимальная разница после повторной загрузки:",
    float(np.max(np.abs(original_prediction - reloaded_prediction))),
)
display(pd.DataFrame({
    "predicción_original": original_prediction,
    "predicción_recargada": reloaded_prediction,
}))

## 29. GPU и резервный режим CPU


In [ ]:
nvidia_binary = сhutil.which("nvidia-smi")
gpu_available = False
gpu_description = "Доступный GPU NVIDIA не обнаружен."
if nvidia_binary:
    gpu_check = сubprocess.run(
        [nvidia_binary, "--query-gpu=name", "--format=csv,noheader"],
        capture_output=True,
        text=True,
        timeout=15,
    )
    gpu_available = gpu_check.returncode == 0 and bool(gpu_check.stdout.strip())
    if gpu_available:
        gpu_description = gpu_check.stdout.strip()

if gpu_available:
    gpu_model = CatBoostRegressor(
        iterations=150,
        depth=7,
        learning_rate=0.05,
        loss_function="RMSE",
        random_seed=SEED,
        verbose=False,
        task_type="GPU",
    )
    gpu_start = perf_counter()
    gpu_model.fit(X_advanced_train, y_advanced_train)
    gpu_seconds = perf_counter() - gpu_start
    cpu_seconds = advanced_comparison.loc[
        advanced_comparison["modelo"] == "CatBoost", "fit_seconds"
    ].iloc[0]
    gpu_comparison = pd.DataFrame({
        "dispositivo": ["CPU", "GPU"],
        "segundos": [cpu_seconds, gpu_seconds],
    })
    display(gpu_comparison)
else:
    gpu_comparison = pd.DataFrame({
        "estado": ["Выполнен резервный режим CPU"],
        "detalle": [gpu_description],
        "modelo_neuronal": ["Вместо PyTorch CUDA использован MLPRegressor"],
    })
    display(gpu_comparison)

print(
    "Ветка GPU использует CatBoost с task_type='GPU', когда оборудование доступно; "
    "на этом компьютере сохранено проверенное выполнение на CPU."
)

## 30. Итоговая сводка и радарная диаграмма по шести критериям


In [ ]:
best_model_row = advanced_comparison.iloc[0]
rmse_min = advanced_comparison["RMSE"].min()
rmse_max = advanced_comparison["RMSE"].max()
quality_score = 1.0 if rmse_max == rmse_min else (
    1 - (best_model_row["RMSE"] - rmse_min) / (rmse_max - rmse_min)
)

inference_min = advanced_comparison["inference_ms_per_row"].min()
inference_max = advanced_comparison["inference_ms_per_row"].max()
inference_score = 1.0 if inference_max == inference_min else (
    1 - (best_model_row["inference_ms_per_row"] - inference_min)
    / (inference_max - inference_min)
)

worst_noise_rmse = noise_summary["mean"].max()
robustness_score = float(np.clip(
    1 - (worst_noise_rmse - baseline_robustness_rmse) / baseline_robustness_rmse,
    0.0,
    1.0,
))
radar_scores = {
    "Качество": float(quality_score),
    "Инференс": float(inference_score),
    "Интерпретируемость": 1.0,
    "Устойчивость": robustness_score,
    "Оптимизация": min(1.0, len(study.trials) / 200),
    "Инженерия": 1.0,
}
radar_frame = pd.DataFrame({
    "criterio": list(radar_scores.keys()),
    "puntuación": list(radar_scores.values()),
})
display(radar_frame)

labels = list(radar_scores.keys())
values = list(radar_scores.values())
angles = np.linspace(0, 2 * np.pi, len(labels), endpoint=False).tolist()
values += values[:1]
angles += angles[:1]
fig, ax = plt.subplots(figsize=(7, 7), сubplot_kw={"polar": True})
ax.plot(angles, values, linewidth=2)
ax.fill(angles, values, alpha=0.25)
ax.set_xticks(angles[:-1])
ax.set_xticklabels(labels)
ax.set_ylim(0, 1)
ax.set_title("Комплексная оценка решения", pad=20)
plt.tight_layout()
plt.show()

In [ ]:
final_summary = advanced_comparison.copy()
final_summary["seleccionado_por_rmse"] = (
    final_summary["modelo"] == final_summary.iloc[0]["modelo"]
)
display(final_summary)
print(f"Модель с наименьшим RMSE: {final_summary.iloc[0]['modelo']}")
print(f"RMSE: {final_summary.iloc[0]['RMSE']:.6f}")
print(f"Конформное покрытие: {conformal_coverage:.3f}")
print(f"Приближённый ATE для has_image: {causal_ate:.6f}")
print(f"Сохранённый конвейер: {pipeline_path.name}")

## 31. Технические выводы

- Решение обрабатывает полный объём данных с помощью Lazy API, потокового выполнения и
  Parquet, а дорогостоящую материализацию ограничивает документированными детерминированными
  выборками.
- Для окончательного сравнения результатов внутренняя тестовая выборка используется только
  один раз. Optuna настраивается на валидационной выборке, выделенной исключительно из
  обучающих данных.
- SHAP, permutation importance, PDP/ICE и LIME описывают поведение модели с глобальной и
  локальной точек зрения.
- Конформное прогнозирование оценивает неопределённость, а изотоническая калибровка исправляет
  связь между прогнозом и наблюдением без подгонки по тестовой выборке.
- Причинно-следственный анализ зависит от предположения об отсутствии ненаблюдаемых
  переменных-смешений. Опровержения проверяют устойчивость, но не превращают наблюдательные
  данные в эксперимент.
- Тесты с шумом, пропусками, OOD и разными начальными зёрнами показывают, когда RMSE
  отклоняется от базового уровня.

Пункт 16, посвящённый подготовке итогового PDF-файла, в этом ноутбуке не выполняется.
